# Push-v3 PPO Split Results Notebook


In [ ]:
from pathlib import Path
import pandas as pd # type: ignore
import numpy as np # type: ignore
import matplotlib.pyplot as plt # type: ignore

ROOT = Path('push_v3_ppo_split_runs')
RESULTS = ROOT / 'results'
AGG_PATH = RESULTS / 'push_v3_aggregate_results.csv'
EP_PATH = RESULTS / 'push_v3_per_episode_results.csv'

# Για re-evaluation results, κάνε uncomment αυτά:
# AGG_PATH = RESULTS / 'eval_final_models' / 'push_v3_eval_summary_by_run.csv'
# EP_PATH = RESULTS / 'eval_final_models' / 'push_v3_eval_raw_episodes.csv'

print('Aggregate:', AGG_PATH)
print('Per episode:', EP_PATH)
agg = pd.read_csv(AGG_PATH)
episodes = pd.read_csv(EP_PATH) if EP_PATH.exists() else None
print('Aggregate shape:', agg.shape)
print('Per-episode shape:', None if episodes is None else episodes.shape)
display(agg.head())
print(agg.columns.tolist())

## Normalize column names

Το training CSV και το re-evaluation CSV έχουν λίγο διαφορετικά column names. Αυτό το cell τα ενώνει σε κοινή μορφή.

In [ ]:
df = agg.copy()

# If using eval script summary, convert long group format to wide train/test format.
if 'group' in df.columns and 'success_rate' in df.columns:
    id_cols = ['config_name', 'split_id', 'split_seed', 'train_seed', 'total_timesteps']
    optional = ['run_name', 'model_path', 'vecnormalize_path']
    id_cols = [c for c in id_cols + optional if c in df.columns]
    wide = df.pivot_table(index=id_cols, columns='group', values=['success_rate', 'avg_return', 'avg_first_success_step'], aggfunc='first')
    wide.columns = [f'{group}_{metric}' for metric, group in wide.columns]
    df = wide.reset_index()

rename_map = {
    'train_success_rate': 'train_success',
    'test_success_rate': 'test_success',
    'train_avg_return': 'train_return',
    'test_avg_return': 'test_return',
    'train_avg_first_success_step': 'train_first_success_step',
    'test_avg_first_success_step': 'test_first_success_step',
}
df = df.rename(columns=rename_map)

if 'success_gap' not in df.columns and {'train_success', 'test_success'} <= set(df.columns):
    df['success_gap'] = df['train_success'] - df['test_success']
if 'return_gap' not in df.columns and {'train_return', 'test_return'} <= set(df.columns):
    df['return_gap'] = df['train_return'] - df['test_return']

display(df.head())
print(df.columns.tolist())

## Experiment overview

In [ ]:
print('Configs:', sorted(df['config_name'].unique()))
print('Splits:', sorted(df['split_id'].unique()))
print('Split seeds:', sorted(df['split_seed'].unique()))
print('Train seeds:', sorted(df['train_seed'].unique()) if 'train_seed' in df.columns else 'N/A')
print('Timesteps:', sorted(df['total_timesteps'].unique()))
print('Rows:', len(df))
cols = ['config_name', 'split_id', 'split_seed', 'train_seed', 'train_success', 'test_success', 'success_gap', 'train_return', 'test_return']
display(df[[c for c in cols if c in df.columns]].sort_values(['config_name', 'split_id']))

## Train vs test success by config/split

In [ ]:
for config in sorted(df['config_name'].unique()):
    d = df[df['config_name'] == config].sort_values('split_id')
    x = np.arange(len(d))
    width = 0.38
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width/2, d['train_success'], width, label='train')
    ax.bar(x + width/2, d['test_success'], width, label='test')
    ax.set_title(f'Push-v3 success rate by split — {config}')
    ax.set_xlabel('Split ID')
    ax.set_ylabel('Success rate')
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(d['split_id'])
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    plt.show()

## 4. Mean success across splits

In [ ]:
summary = (
    df.groupby('config_name')
    .agg(
        train_success_mean=('train_success', 'mean'),
        train_success_std=('train_success', 'std'),
        test_success_mean=('test_success', 'mean'),
        test_success_std=('test_success', 'std'),
        train_return_mean=('train_return', 'mean'),
        test_return_mean=('test_return', 'mean'),
        success_gap_mean=('success_gap', 'mean'),
        success_gap_std=('success_gap', 'std'),
        runs=('split_id', 'count'),
    )
    .reset_index()
)
display(summary)

x = np.arange(len(summary))
width = 0.38
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width/2, summary['train_success_mean'], width, yerr=summary['train_success_std'].fillna(0), capsize=4, label='train')
ax.bar(x + width/2, summary['test_success_mean'], width, yerr=summary['test_success_std'].fillna(0), capsize=4, label='test')
ax.set_title('Push-v3 mean success across splits')
ax.set_xlabel('PPO config')
ax.set_ylabel('Mean success rate')
ax.set_ylim(0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(summary['config_name'], rotation=20, ha='right')
ax.grid(True, axis='y', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Success gap: train - test

In [ ]:
plot_df = df.sort_values(['config_name', 'split_id']).reset_index(drop=True)
labels = plot_df['config_name'] + '\nsplit ' + plot_df['split_id'].astype(str)
x = np.arange(len(plot_df))
fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(x, plot_df['success_gap'])
ax.axhline(0, linestyle='--')
ax.set_title('Push-v3 train-test success gap by run')
ax.set_xlabel('Run')
ax.set_ylabel('Success gap')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
display(plot_df[['config_name', 'split_id', 'train_success', 'test_success', 'success_gap']])

## First success step

Lower is better

In [ ]:
first_cols = {'train_first_success_step', 'test_first_success_step'}
if first_cols <= set(df.columns):
    fs_summary = df.groupby('config_name').agg(
        train_first_success_step_mean=('train_first_success_step', 'mean'),
        test_first_success_step_mean=('test_first_success_step', 'mean'),
    ).reset_index()
    display(fs_summary)
    x = np.arange(len(fs_summary))
    width = 0.38
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x - width/2, fs_summary['train_first_success_step_mean'], width, label='train')
    ax.bar(x + width/2, fs_summary['test_first_success_step_mean'], width, label='test')
    ax.set_title('Push-v3 average first success step')
    ax.set_xlabel('PPO config')
    ax.set_ylabel('Avg first success step')
    ax.set_xticks(x)
    ax.set_xticklabels(fs_summary['config_name'], rotation=20, ha='right')
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('First-success columns not found in this CSV.')

## Per-episode distributions

In [ ]:
if episodes is not None:
    display(episodes.head())
    print(episodes.columns.tolist())
    config_col = 'config_name' if 'config_name' in episodes.columns else 'config'
    ep_summary = episodes.groupby([config_col, 'split_id', 'group'])['success'].mean().reset_index()
    display(ep_summary.head(30))
else:
    print('No per-episode CSV found.')

## Save figures

In [ ]:
fig_dir = ROOT / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(summary))
width = 0.38
ax.bar(x - width/2, summary['train_success_mean'], width, yerr=summary['train_success_std'].fillna(0), capsize=4, label='train')
ax.bar(x + width/2, summary['test_success_mean'], width, yerr=summary['test_success_std'].fillna(0), capsize=4, label='test')
ax.set_title('Push-v3 mean success across splits')
ax.set_xlabel('PPO config')
ax.set_ylabel('Mean success rate')
ax.set_ylim(0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(summary['config_name'], rotation=20, ha='right')
ax.grid(True, axis='y', alpha=0.3)
ax.legend()
plt.tight_layout()
out1 = fig_dir / 'push_v3_mean_success_across_splits.png'
fig.savefig(out1, dpi=200, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(plot_df))
ax.bar(x, plot_df['success_gap'])
ax.axhline(0, linestyle='--')
ax.set_title('Push-v3 train-test success gap by run')
ax.set_xlabel('Run')
ax.set_ylabel('Success gap')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
out2 = fig_dir / 'push_v3_success_gap_by_run.png'
fig.savefig(out2, dpi=200, bbox_inches='tight')
plt.close(fig)
print('Saved:')
print(out1)
print(out2)